<a href="https://colab.research.google.com/github/kabir01chamuah-alt/project-file-and-folder-organiser/blob/main/project_file_and_folder_organiser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import hashlib
import shutil
from collections import defaultdict


# ==========================================
# A6: DUPLICATE FILE FINDER & ORGANISER
# ==========================================

# Configuration
FOLDER_PATH = "/content/TestFolder"

# False = Dry Run (safe mode)
# True = Actually move duplicate files
MOVE_DUPLICATES = True


# ==========================================
# 1. CALCULATE SHA-256 HASH
# ==========================================

def calculate_hash(filepath):

    sha256 = hashlib.sha256()

    try:
        with open(filepath, "rb") as file:

            while True:
                chunk = file.read(8 * 1024)

                if not chunk:
                    break

                sha256.update(chunk)

        return sha256.hexdigest()

    except OSError as e:
        print("Error reading file:", filepath, e)
        return None


# ==========================================
# 2. SCAN DIRECTORY AND FIND DUPLICATES
# ==========================================

def find_duplicates(folder):

    file_groups = defaultdict(list)
    size_groups = defaultdict(list)

    print("\n========== SCANNING DIRECTORY ==========\n")

    for root, dirs, files in os.walk(folder):

        # Avoid scanning the Duplicates folder
        dirs[:] = [
            d for d in dirs
            if d != "Duplicates"
            and not os.path.islink(os.path.join(root, d))
        ]

        for filename in files:

            filepath = os.path.join(root, filename)

            if os.path.islink(filepath):
                continue

            try:
                size = os.path.getsize(filepath)

                print("File:", filepath)
                print("Size:", size, "bytes\n")

                # Group files by size
                size_groups[size].append(filepath)

            except OSError as e:
                print("Error accessing file:", e)

    # Calculate hash only for files of matching sizes
    print("========== CALCULATING HASHES ==========\n")

    for size, paths in size_groups.items():

        if len(paths) < 2:
            continue

        for filepath in paths:

            file_hash = calculate_hash(filepath)

            if file_hash is not None:
                file_groups[file_hash].append(filepath)

    # Keep only groups with duplicates
    duplicates = {}

    for file_hash, paths in file_groups.items():

        if len(paths) > 1:
            duplicates[file_hash] = paths

    return duplicates


# ==========================================
# 3. DISPLAY DUPLICATE REPORT
# ==========================================

def display_report(duplicates):

    print("\n========== DUPLICATE FILE REPORT ==========\n")

    if not duplicates:
        print("No duplicate files found.")
        return

    total_duplicates = 0

    for number, (file_hash, paths) in enumerate(
        duplicates.items(), start=1
    ):

        print("Duplicate Group:", number)
        print("SHA-256 Hash:", file_hash)

        for path in paths:
            print("  ", path)

        total_duplicates += len(paths) - 1

        print()

    print("Total duplicate groups:", len(duplicates))
    print("Total duplicate copies:", total_duplicates)


# ==========================================
# 4. ORGANISE DUPLICATE FILES
# ==========================================

def organise_duplicates(duplicates, folder, dry_run=True):

    duplicate_folder = os.path.join(folder, "Duplicates")

    print("\n========== ORGANISING FILES ==========\n")

    for paths in duplicates.values():

        # Keep the first file as original
        # Move the remaining copies

        for filepath in paths[1:]:

            relative_path = os.path.relpath(
                filepath, folder
            )

            destination = os.path.join(
                duplicate_folder, relative_path
            )

            if dry_run:

                print("[DRY RUN] Would move:", filepath)
                print("          To:", destination)
                print()

                continue

            try:

                os.makedirs(
                    os.path.dirname(destination),
                    exist_ok=True
                )

                # Prevent overwriting existing files
                if os.path.exists(destination):

                    name, extension = os.path.splitext(
                        destination
                    )

                    counter = 1

                    while os.path.exists(
                        f"{name}_{counter}{extension}"
                    ):
                        counter += 1

                    destination = (
                        f"{name}_{counter}{extension}"
                    )

                shutil.move(filepath, destination)

                print("Moved:", filepath)
                print("To:", destination)
                print()

            except (OSError, shutil.Error) as e:

                print("Error moving file:", e)


# ==========================================
# 5. MAIN FUNCTION
# ==========================================

def main():

    folder = os.path.abspath(FOLDER_PATH)

    print("========================================")
    print("     DUPLICATE FILE FINDER")
    print("========================================")

    if not os.path.isdir(folder):

        print("Error: Folder does not exist!")
        print("Please create the folder:", folder)
        return

    # Find duplicate files
    duplicates = find_duplicates(folder)

    # Display duplicate report
    display_report(duplicates)

    # Organise duplicate files
    if duplicates:

        if MOVE_DUPLICATES:

            print("\nACTUAL MOVE MODE SELECTED")

            organise_duplicates(
                duplicates,
                folder,
                dry_run=False
            )

        else:

            print("\nSAFE DRY-RUN MODE SELECTED")

            organise_duplicates(
                duplicates,
                folder,
                dry_run=True
            )

    print("\n========================================")
    print("          PROCESS COMPLETED")
    print("========================================")


# ==========================================
# 6. EXECUTE PROGRAM
# ==========================================

main()

     DUPLICATE FILE FINDER

========== SCANNING DIRECTORY ==========

File: /content/TestFolder/unique.txt
Size: 22 bytes

File: /content/TestFolder/file1.txt
Size: 29 bytes

File: /content/TestFolder/file2.txt
Size: 29 bytes

File: /content/TestFolder/Subfolder/copy.txt
Size: 29 bytes

========== CALCULATING HASHES ==========


========== DUPLICATE FILE REPORT ==========

Duplicate Group: 1
SHA-256 Hash: 93c127ac612105b9870b73e33b6f12806908a23dc8f4f17e24661798be90e258
   /content/TestFolder/file1.txt
   /content/TestFolder/file2.txt
   /content/TestFolder/Subfolder/copy.txt

Total duplicate groups: 1
Total duplicate copies: 2

ACTUAL MOVE MODE SELECTED

========== ORGANISING FILES ==========

Moved: /content/TestFolder/file2.txt
To: /content/TestFolder/Duplicates/file2.txt

Moved: /content/TestFolder/Subfolder/copy.txt
To: /content/TestFolder/Duplicates/Subfolder/copy.txt


          PROCESS COMPLETED


In [ ]:
!ls -R /content/TestFolder

/content/TestFolder:
Duplicates  file1.txt  Subfolder  unique.txt

/content/TestFolder/Duplicates:
file2.txt  Subfolder

/content/TestFolder/Duplicates/Subfolder:
copy.txt

/content/TestFolder/Subfolder:


In [ ]:
import os

# Create the main folder
folder = "/content/TestFolder"

os.makedirs(folder + "/Subfolder", exist_ok=True)

# Create first file
with open(folder + "/file1.txt", "w") as f:
    f.write("Hello, this is my AI project!")

# Create duplicate file
with open(folder + "/file2.txt", "w") as f:
    f.write("Hello, this is my AI project!")

# Create another duplicate in subfolder
with open(folder + "/Subfolder/copy.txt", "w") as f:
    f.write("Hello, this is my AI project!")

# Create a unique file
with open(folder + "/unique.txt", "w") as f:
    f.write("This is a unique file.")

print("Test folder created successfully!")
print("All test files created successfully!")

Test folder created successfully!
All test files created successfully!
